# 02 — Influence Math Debug (synthetic data)

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
D, V = 10, 200
raw = rng.random((D, V)).astype(np.float32)
grads = raw / np.linalg.norm(raw, axis=1, keepdims=True)

efficient = grads @ grads.mean(axis=0)
pairwise  = np.array([(1/D)*(grads[i] @ grads.T).sum() for i in range(D)])

print("max abs diff:", np.abs(efficient - pairwise).max())
assert np.allclose(efficient, pairwise, rtol=1e-5), "identity FAILED"
print("identity OK")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from influence_curriculum.score import InfluenceConfig, compute_influence_matrix

MODEL = "sshleifer/tiny-gpt2"  # 2-layer toy model, downloads fast
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)

synthetic_texts = ["The cat sat on the mat.", "Dogs are great pets.", "I love machine learning."]
encodings = [tokenizer(t, return_tensors="pt") for t in synthetic_texts]

# Save a fake checkpoint by just saving the untrained model
import tempfile, os
tmp = tempfile.mkdtemp()
model.save_pretrained(tmp)

Phi = compute_influence_matrix([tmp], encodings, InfluenceConfig(), device="cpu")
print("Phi shape:", Phi.shape)   # (3, 1)
print("Phi values:", Phi)

In [ ]:
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

plt.hist(Phi[:, 0], bins=20)
plt.xlabel("influence score"); plt.ylabel("count")
plt.title("Influence distribution (synthetic)")
plt.show()